# API Project - Bus Bunching
### Multi-agency Comparsion (WMATA, MBTA, MTA)
Question: What route and temporal factors are associated with bus bunching on high-frequency corridors, and do these relationships vary across different transit agencies (NYC, Boston, and DC)?

In [1]:
import pandas as pd
import numpy as np
import requests
import ratelim
import tenacity

### 1) WMATA API
#### Bus Routes: D40, D60, D80, C53, C61, D20

In [47]:
WMATA_KEY = "5ec08fcd21744590b1a0f655710c4c91"

In [48]:
wmata_url = "https://api.wmata.com/Bus.svc/json/jBusPositions"
wmata_params = {"RouteID": "D40"}  
wmata_headers = {"api_key": WMATA_KEY}

requests.get(wmata_url, params=wmata_params, headers=wmata_headers)

<Response [200]>

In [49]:
wmata_response = requests.get(wmata_url, params=wmata_params, headers=wmata_headers)
wmata_response.json()

{'BusPositions': [{'VehicleID': '5503',
   'Lat': 38.89225834813611,
   'Lon': -77.02388763427734,
   'Deviation': 5.0,
   'DateTime': '2026-03-25T11:40:42',
   'TripID': '27051040',
   'RouteID': 'D40',
   'DirectionNum': 1,
   'DirectionText': 'SOUTH',
   'TripHeadsign': 'ARCHIVES',
   'TripStartTime': '2026-03-25T10:36:00',
   'TripEndTime': '2026-03-25T11:35:00',
   'BlockNumber': 'M610'},
  {'VehicleID': '5501',
   'Lat': 38.98890902545001,
   'Lon': -77.02657318115234,
   'Deviation': 1.0,
   'DateTime': '2026-03-25T11:40:53',
   'TripID': '3968040',
   'RouteID': 'D40',
   'DirectionNum': 0,
   'DirectionText': 'NORTH',
   'TripHeadsign': 'SILVER SPRING',
   'TripStartTime': '2026-03-25T10:48:00',
   'TripEndTime': '2026-03-25T11:45:00',
   'BlockNumber': 'M618'},
  {'VehicleID': '5500',
   'Lat': 38.96142444095096,
   'Lon': -77.02799987792969,
   'Deviation': 2.0,
   'DateTime': '2026-03-25T11:41:03',
   'TripID': '28469040',
   'RouteID': 'D40',
   'DirectionNum': 0,
   'Dire

#### Vars Used: 
- `Deviation` — the primary outcome, minutes late/early so no GTFS needed
- `DateTime` — timestamp will be used for peak hour flag
- `RouteID`, `DirectionNum`, `VehicleID`, `TripID` — route/trip identifiers (`DirectionNum`, 0=North 1=South)
- `TripStartTime` — scheduled headway, will be used to calc the gap between consecutive trips
- `Lat`, `Lon` — Vehicle position

In [50]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

wmata_positions = response.json()['BusPositions']

df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
df['agency'] = 'WMATA'

print(df)
print(df.dtypes) #Checks for var types (timestamp and trip_start_time)

  vehicle_id        lat        lon  deviation            timestamp   trip_id  \
0       5486  38.898360 -76.946907        4.0  2026-03-25T08:24:04  24700040   
1       5484  38.895509 -76.951414        5.0  2026-03-25T08:32:43  19623040   
2       7179  38.899818 -77.026924       -1.0  2026-03-25T08:32:37  33845040   
3       7197  38.898257 -76.972305       12.0  2026-03-25T08:32:43  34760040   
4       5489  38.900219 -77.012070        5.0  2026-03-25T08:32:48  11314040   
5       5531  38.898602 -76.974260        2.0  2026-03-25T08:32:40   2210040   
6       5482  38.899818 -77.020283        2.0  2026-03-25T08:32:47  29055040   
7       5491  38.900219 -77.036461       -1.0  2026-03-25T08:32:31  20607040   
8       5526  38.901615 -77.038597        0.0  2026-03-25T08:32:31   3186040   

  route_id  direction      trip_start_time agency  
0      D20          0  2026-03-25T07:42:00  WMATA  
1      D20          0  2026-03-25T07:52:00  WMATA  
2      D20          1  2026-03-25T08:00:00 

In [51]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
print(df.dtypes)

vehicle_id                 object
lat                       float64
lon                       float64
deviation                 float64
timestamp          datetime64[ns]
trip_id                    object
route_id                   object
direction                   int64
trip_start_time    datetime64[ns]
agency                     object
dtype: object


In [58]:
routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20']
dfs = []

for route in routes:
    wmata_response = requests.get(wmata_url, params={"RouteID": route}, headers=wmata_headers)
    wmata_positions = wmata_response.json()['BusPositions']
    df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
    df['agency'] = 'WMATA'
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
    dfs.append(df)

wmata_df = pd.concat(dfs, ignore_index=True)

In [59]:
wmata_df

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency
0,5503,38.892258,-77.023888,5.0,2026-03-25 11:44:43,27051040,D40,1,2026-03-25 10:36:00,WMATA
1,5501,38.993723,-77.030403,1.0,2026-03-25 11:45:14,3968040,D40,0,2026-03-25 10:48:00,WMATA
2,5500,38.967221,-77.027505,3.0,2026-03-25 11:45:28,28469040,D40,0,2026-03-25 11:00:00,WMATA
3,5506,38.914144,-77.021927,2.0,2026-03-25 11:45:33,11603040,D40,1,2026-03-25 11:00:00,WMATA
4,5521,38.946060,-77.026601,1.0,2026-03-25 11:45:21,6977040,D40,0,2026-03-25 11:12:00,WMATA
...,...,...,...,...,...,...,...,...,...,...
60,7168,38.900311,-77.004921,5.0,2026-03-25 11:45:35,17285040,D20,1,2026-03-25 11:20:00,WMATA
61,5490,38.899327,-77.013990,1.0,2026-03-25 11:45:35,2665040,D20,0,2026-03-25 11:30:00,WMATA
62,5484,38.900206,-76.985914,2.0,2026-03-25 11:45:11,14753040,D20,1,2026-03-25 11:30:00,WMATA
63,5488,38.899830,-77.027855,-1.0,2026-03-25 11:45:21,21317040,D20,0,2026-03-25 11:40:00,WMATA


In [61]:
print(wmata_df['route_id'].unique()) #Checking that all the routes i need are in the df

['D40' 'D60' 'D80' 'C53' 'C61' 'D20']


### 2) MBTA API
#### Bus Routes: Route 1, 23, 28, 39, 57, 66

In [42]:
MBTA_KEY = '4aa18b9edd1341e383dc8fe55a788cca'

In [44]:
#Test
mbta_url = "https://api-v3.mbta.com/vehicles"
mbta_params = {"filter[route]": "1"}
mbta_headers = {"x-api-key": MBTA_KEY}

requests.get(mbta_url, params=mbta_params, headers=mbta_headers)

<Response [200]>

In [45]:
mbta_response = requests.get(mbta_url, params=mbta_params, headers=mbta_headers)
mbta_response.json()

{'data': [{'attributes': {'bearing': 312,
    'carriages': [],
    'current_status': 'IN_TRANSIT_TO',
    'current_stop_sequence': 8,
    'direction_id': 0,
    'label': '1894',
    'latitude': 42.336149,
    'longitude': -71.076452,
    'occupancy_status': 'MANY_SEATS_AVAILABLE',
    'revenue': 'REVENUE',
    'speed': None,
    'updated_at': '2026-03-25T11:33:19-04:00'},
   'id': 'y1894',
   'links': {'self': '/vehicles/y1894'},
   'relationships': {'route': {'data': {'id': '1', 'type': 'route'}},
    'stop': {'data': {'id': '10590', 'type': 'stop'}},
    'trip': {'data': {'id': '73604101', 'type': 'trip'}}},
   'type': 'vehicle'},
  {'attributes': {'bearing': 90,
    'carriages': [],
    'current_status': 'STOPPED_AT',
    'current_stop_sequence': 1,
    'direction_id': 0,
    'label': '1883',
    'latitude': 42.32977,
    'longitude': -71.08375,
    'occupancy_status': 'MANY_SEATS_AVAILABLE',
    'revenue': 'REVENUE',
    'speed': None,
    'updated_at': '2026-03-25T11:32:55-04:00'}

#### Vars Used
- `id` — vehicle_id
- `attributes.latitude`, `attributes.longitude` — lat, lon
- `attributes.direction_id` — direction (already 0/1, but doesn't tell me which direction that is)
- `attributes.updated_at` — timestamp
- `attributes.current_stop_sequence` — stop_sequence (not available in WMATA)
- `relationships.trip.data.id` — trip_id
- `relationships.route.data.id` — route_id

#### Notes
- No deviation field — must be computed from GTFS static feed in R
- I'll also need GTFS static for the direction north or south bound
- This nested unlike WMATA which was more straight forward

In [ ]:
mbta_fields = {

### 3) MTA API
#### Bus Routes: B44, B46, B41, Bx12, M14A, M104A

In [ ]:
MTA_KEY = '7027c575-3c63-477c-91d8-3d404472e8a0'

In [ ]:
mta_url = 
mta_params = 
mta_headers = 